## Sales data

##### Import packages

In [ ]:
import pandas as pd
import fsspec
import s3fs
import hashlib
import numpy as np
import pyarrow

##### Read customer Dimension files

In [ ]:
customer_df = pd.read_csv('s3://retail-data-142083400213/dimension/customer.csv')
customer_df['customer_id'] = customer_df.apply(
    lambda row: hashlib.md5(
        (str(row['name']) + str(row['city']) + str(row['state'])).encode()
    ).hexdigest(),
    axis = 1
)
customer_df

##### Read the product dimension file

In [ ]:
product_df = pd.read_csv('s3://retail-data-142083400213/dimension/products.csv')
product_df['product_id'] = product_df.apply(
    lambda row: hashlib.md5(
        (str(row['product_name']) + str(row['brand'])).encode()
    ).hexdigest(),
    axis = 1
)
product_df

##### Read the store dimension file

In [ ]:
store_df = pd.read_csv('s3://retail-data-142083400213/dimension/stores.csv')
store_df['store_id'] = store_df.apply(
    lambda row: hashlib.md5(
        (str(row['store_name'])).encode()
    ).hexdigest(),
    axis = 1
)
store_df

##### Read the sales data

In [ ]:
sales_df = pd.read_csv('s3://retail-data-142083400213/sales/sales.csv')
# sales_df = sales_df.merge(
#     customer_df,
#     left_on=['name', 'city', 'region'],
#     right_on=['name', 'city', 'state'],
#     how='left'
# )
# sales_df = sales_df[['invoice_no', 'customer_id', 'product', 'brand', 'store', 'quantity', 'unit_price', 'discount', 'order_timestamp', 'payment_mode']]
# sales_df

sales_df = (
    sales_df
    .merge(
        customer_df,
        left_on=['name', 'city', 'region'],
        right_on=['name', 'city', 'state'],
        how='left'
    )
    .merge(
        product_df,
        left_on=['product', 'brand'],
        right_on=['product_name', 'brand'],
        how='left'
    )
    .merge(
        store_df,
        left_on=['store'],
        right_on=['store_name'],
        how='left'
    )
    .loc[:, [
        'invoice_no',
        'customer_id',
        'product_id',
        'store_id',
        'cost_price',
        'quantity',
        'unit_price',
        'discount',
        'order_timestamp',
        'payment_mode'
    ]]
)
sales_df

In [ ]:
sales_df['gross_amount'] = sales_df['quantity'] * sales_df['unit_price']
sales_df['net_amount'] = sales_df['gross_amount'] - sales_df['discount']
sales_df

In [ ]:
sales_df['profit'] = sales_df['net_amount'] - (sales_df['cost_price'] * sales_df['quantity'])
sales_df['profit_percentage'] = (sales_df['profit'] / sales_df['net_amount']) * 100
sales_df

In [ ]:
conditions = [
    sales_df['net_amount'] > 50000,
    sales_df['net_amount'] > 10000
]

choices = ['High Value', 'Medium Value']
sales_df['segment'] = np.select(conditions, choices, default='Low Value')
sales_df

In [ ]:
# high_value_sales = sales_df[sales_df['segment'] == 'High Value']
high_value_sales = sales_df.query("segment == 'High Value'")
high_value_sales[['invoice_no']]

In [ ]:
high_value_sales = sales_df.query("profit_percentage > 30")
high_margin_products = (
    high_value_sales
    .merge(product_df, on=['product_id'], how='left')
    .loc[:,[
        'product_name',
        'brand',
        'category'
    ]]
)
high_margin_products

In [ ]:
'''
CSV : Comma Separated Values
- Row based data storage format
- File size can be large, but not suitable for very large datasets
- Easy to human readable and write, widely supported
- Reading a large CSV file can be memory intensive and slow, especially if the file is larger than available memory
Parquet : Columnar storage format
- Column based data storage format
- More efficient for large datasets, supports compression and encoding
- Not human readable, but optimized for query performance and storage efficiency
'''

In [ ]:
# Write the dataframe to Parquet format in S3
sales_df.to_parquet('s3://retail-data-142083400213/output/sales.parquet',
                    engine='pyarrow',
                    index=False
                )

In [ ]:
sales_df.to_csv('s3://retail-data-142083400213/output/sales.csv',
                index=False)